# Stress Prediction v23

Targeted improvements over v22 (LB 0.38495):

- **Fix 1 — Per-class alpha calibration**: Optimize `alpha_0`, `alpha_1`, `alpha_2` separately using LOPO OOF probabilities to avoid class-1 collapse.
- **Fix 2 — Temporal context**: Session position/elapsed, lag window features (3–6 min before current window).
- **Fix 3 — Honest LOPO CV**: Replaces leaky StratifiedKFold for calibration.
- **Bonus — XGBoost blend**: Adds model diversity (60% LGBM / 40% XGB if available).

## 0. Install Dependencies

In [1]:
# Install required libraries
%pip install numpy pandas scipy scikit-learn lightgbm

# Optional: XGBoost for ensemble blending (60% LGBM / 40% XGB)
%pip install xgboost


[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 26.0 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. Imports & Global Config

In [2]:
#!/usr/bin/env python3
import warnings
import shutil
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from scipy import stats as spstats
from scipy import signal as sps
from scipy.integrate import trapezoid
from scipy.optimize import minimize

from sklearn.impute import SimpleImputer
from sklearn.metrics import balanced_accuracy_score
from sklearn.model_selection import StratifiedKFold, LeaveOneGroupOut

import lightgbm as lgb

try:
    import xgboost as xgb
    HAS_XGB = True
    print("✓ XGBoost available — will blend LGBM + XGB")
except ImportError:
    HAS_XGB = False
    print("✗ XGBoost not available — using LightGBM only")

warnings.filterwarnings('ignore')
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

✓ XGBoost available — will blend LGBM + XGB


## 2. Load & Clean Data

In [3]:
DATA_DIR    = Path('.')
TRAIN_DATA  = pd.read_csv(DATA_DIR / 'train-sensor.csv')
TRAIN_LABEL = pd.read_csv(DATA_DIR / 'train-label.csv')
TEST_DATA   = pd.read_csv(DATA_DIR / 'test-sensor.csv')
TEST_LABEL  = pd.read_csv(DATA_DIR / 'test-label.csv')

print('Raw shapes:')
print(f'  TRAIN_DATA : {TRAIN_DATA.shape}')
print(f'  TRAIN_LABEL: {TRAIN_LABEL.shape}')
print(f'  TEST_DATA  : {TEST_DATA.shape}')
print(f'  TEST_LABEL : {TEST_LABEL.shape}')

SENSOR_COLS = ['accel_x', 'accel_y', 'accel_z', 'eda', 'heart_rate', 'temperature']


def clean_sensor(df):
    out = df.copy()
    out['pid'] = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    for c in SENSOR_COLS:
        out[c] = pd.to_numeric(out[c], errors='coerce').astype(float)
    out['accel_x']     = out['accel_x'].clip(-128, 127)
    out['accel_y']     = out['accel_y'].clip(-128, 127)
    out['accel_z']     = out['accel_z'].clip(-128, 127)
    out['eda']         = out['eda'].clip(0, 60)
    out['heart_rate']  = out['heart_rate'].clip(40, 190)
    out['temperature'] = out['temperature'].clip(20, 40)
    return out.sort_values(['pid', 'timestamp']).reset_index(drop=True)


def clean_label(df):
    out = df.copy()
    out['id']        = pd.to_numeric(out['id'], errors='raise').astype(int)
    out['pid']       = out['pid'].astype(str)
    out['timestamp'] = pd.to_numeric(out['timestamp'], errors='coerce').astype(float)
    out['stress']    = pd.to_numeric(out['stress'], errors='coerce')
    return out


TRAIN_DATA  = clean_sensor(TRAIN_DATA)
TEST_DATA   = clean_sensor(TEST_DATA)
TRAIN_LABEL = clean_label(TRAIN_LABEL)
TEST_LABEL  = clean_label(TEST_LABEL)
print('Cleaned ✓')

Raw shapes:
  TRAIN_DATA : (4694400, 8)
  TRAIN_LABEL: (815, 4)
  TEST_DATA  : (5921280, 8)
  TEST_LABEL : (1028, 4)
Cleaned ✓


## 3. Resting Baseline per Subject

In [4]:
def compute_resting_baselines(sensor_df, low_pct=10):
    """Identify bottom low_pct% of arousal stream as personal rest baseline."""
    refs = {}
    for pid, grp in sensor_df.groupby('pid'):
        hr   = grp['heart_rate'].values.astype(float)
        eda  = grp['eda'].values.astype(float)
        temp = grp['temperature'].values.astype(float)
        valid = np.isfinite(hr) & np.isfinite(eda)
        if valid.sum() < 100:
            refs[pid] = {
                'hr': float(np.nanmedian(hr)), 'eda': float(np.nanmedian(eda)),
                'temp': float(np.nanmedian(temp)), 'hr_std': 5.0, 'eda_std': 0.5
            }
            continue
        hr_v, eda_v, temp_v = hr[valid], eda[valid], temp[valid]
        hr_z  = (hr_v  - hr_v.mean())  / (hr_v.std()  + 1e-9)
        eda_z = (eda_v - eda_v.mean()) / (eda_v.std() + 1e-9)
        arousal = hr_z + eda_z
        thr = np.percentile(arousal, low_pct)
        mask = arousal < thr
        if mask.sum() < 10:
            mask = np.ones(len(arousal), dtype=bool)
        refs[pid] = {
            'hr':      float(np.median(hr_v[mask])),
            'eda':     float(np.median(eda_v[mask])),
            'temp':    float(np.median(temp_v[mask])),
            'hr_std':  float(np.std(hr_v[mask])  + 1e-3),
            'eda_std': float(np.std(eda_v[mask]) + 1e-3),
        }
    return refs


TRAIN_REFS = compute_resting_baselines(TRAIN_DATA)
TEST_REFS  = compute_resting_baselines(TEST_DATA)
print(f'Resting baselines computed for {len(TRAIN_REFS)} train / {len(TEST_REFS)} test PIDs ✓')

Resting baselines computed for 7 train / 8 test PIDs ✓


## 4. Session Temporal Enrichment *(NEW)*

Adds `session_elapsed_ms`, `session_position` (0–1), `session_label_idx`, `session_total_labels` to each label row.

In [5]:
GAP_MS = 30 * 60 * 1000  # 30-min gap → new session


def enrich_labels_with_session_info(label_df):
    out = label_df.copy()
    out['session_elapsed_ms']   = np.nan
    out['session_position']     = np.nan
    out['session_label_idx']    = np.nan
    out['session_total_labels'] = np.nan

    for pid, grp in out.groupby('pid'):
        grp_s = grp.sort_values('timestamp')
        ts    = grp_s['timestamp'].values
        breaks = np.where(np.diff(ts) > GAP_MS)[0] + 1
        bounds = np.r_[0, breaks, len(ts)]

        for i in range(len(bounds) - 1):
            s, e   = bounds[i], bounds[i + 1]
            idx    = grp_s.index[s:e]
            sess_ts = ts[s:e]
            elapsed = (sess_ts - sess_ts[0]).astype(float)
            dur     = float(elapsed[-1]) if len(elapsed) > 1 else 1.0

            out.loc[idx, 'session_elapsed_ms']   = elapsed
            out.loc[idx, 'session_position']     = elapsed / max(dur, 1.0)
            out.loc[idx, 'session_label_idx']    = np.arange(e - s, dtype=float)
            out.loc[idx, 'session_total_labels'] = float(e - s)

    return out


TRAIN_LABEL_E = enrich_labels_with_session_info(TRAIN_LABEL)
TEST_LABEL_E  = enrich_labels_with_session_info(TEST_LABEL)
print('Session temporal info added ✓')

Session temporal info added ✓


## 5. Feature Extraction

v22 features + session temporal + lag window (3–6 min before) + extra percentiles p10/p90.

In [6]:
WINDOW_MS = 180_000   # 3-min main window
HALF_MS   =  90_000
THIRD_MS  =  60_000
SHORT_MS  =  60_000   # 1-min fast-response window
LAG_MS    = 180_000   # lag offset: 3–6 min before current window


def hrv_time_domain(bpm_series):
    f = {}
    bpm = bpm_series.dropna().values.astype(float)
    if len(bpm) < 10:
        for k in ['sdnn', 'rmssd', 'pnn25', 'pnn50', 'mean_rr', 'cv_rr']:
            f['hrv_' + k] = np.nan
        return f
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    rr       = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_diff  = np.diff(rr)
    f['hrv_sdnn']    = float(np.std(rr))
    f['hrv_rmssd']   = float(np.sqrt(np.mean(rr_diff**2))) if len(rr_diff) else 0.0
    f['hrv_pnn25']   = float(np.mean(np.abs(rr_diff) > 25) * 100) if len(rr_diff) else 0.0
    f['hrv_pnn50']   = float(np.mean(np.abs(rr_diff) > 50) * 100) if len(rr_diff) else 0.0
    f['hrv_mean_rr'] = float(np.mean(rr))
    f['hrv_cv_rr']   = f['hrv_sdnn'] / f['hrv_mean_rr'] if f['hrv_mean_rr'] > 1e-6 else 0.0
    return f


def hrv_frequency_domain(bpm_series):
    f = {'hrv_lf': np.nan, 'hrv_hf': np.nan, 'hrv_lf_hf': np.nan, 'hrv_total_power': np.nan}
    bpm = bpm_series.dropna().values.astype(float)
    bpm_1hz = bpm[::32] if len(bpm) >= 32 else bpm
    if len(bpm_1hz) < 30:
        return f
    rr = 60000.0 / np.clip(bpm_1hz, 30, 220)
    rr_c = rr - rr.mean()
    nperseg = min(len(rr_c), 64)
    if nperseg < 16:
        return f
    try:
        freqs, psd = sps.welch(rr_c, fs=1.0, nperseg=nperseg, noverlap=nperseg // 2)
        def bp(lo, hi):
            m = (freqs >= lo) & (freqs < hi)
            return float(trapezoid(psd[m], freqs[m])) if m.sum() >= 2 else 0.0
        f['hrv_lf']          = bp(0.04, 0.15)
        f['hrv_hf']          = bp(0.15, 0.40)
        f['hrv_total_power'] = bp(0.0033, 0.40)
        f['hrv_lf_hf']       = f['hrv_lf'] / (f['hrv_hf'] + 1e-6)
    except Exception:
        pass
    return f


def eda_peak_features(eda_series):
    f = {
        'eda_n_peaks': np.nan, 'eda_peaks_per_min': np.nan,
        'eda_mean_prominence': np.nan, 'eda_max_prominence': np.nan,
    }
    eda = eda_series.dropna().values.astype(float)
    if len(eda) < 40:
        return f
    eda_4hz = eda[::8] if len(eda) >= 100 else eda
    if len(eda_4hz) < 16:
        return f
    try:
        wl = min(15, len(eda_4hz) // 2 * 2 + 1)
        phasic = (eda_4hz - sps.savgol_filter(eda_4hz, wl, 2) + np.mean(eda_4hz)
                  if wl >= 5 else eda_4hz)
        peaks, props = sps.find_peaks(phasic, prominence=0.02, distance=4, width=1)
        f['eda_n_peaks'] = float(len(peaks))
        dur_min = len(eda_4hz) / (4.0 * 60.0)
        f['eda_peaks_per_min'] = float(len(peaks) / dur_min) if dur_min > 0 else 0.0
        if len(peaks) > 0:
            f['eda_mean_prominence'] = float(np.mean(props['prominences']))
            f['eda_max_prominence']  = float(np.max(props['prominences']))
        else:
            f['eda_mean_prominence'] = f['eda_max_prominence'] = 0.0
    except Exception:
        pass
    return f

In [7]:
def extract_features(label_df_e, sensor_df, pid_enc_map, refs):
    """Extract all features for label_df_e (must be the session-enriched version)."""
    sensor_by_pid = {
        pid: grp.sort_values('timestamp').reset_index(drop=True)
        for pid, grp in sensor_df.groupby('pid')
    }
    rows = []

    for n, lrow in enumerate(label_df_e.itertuples(index=False), 1):
        pid = lrow.pid
        ts  = float(lrow.timestamp)
        lid = int(lrow.id)
        feat = {'id': lid}

        # ── Session temporal features (NEW) ────────────────────────────────
        for col in ['session_elapsed_ms', 'session_position',
                    'session_label_idx', 'session_total_labels']:
            val = getattr(lrow, col)
            feat[col] = float(val) if np.isfinite(float(val)) else np.nan

        sg = sensor_by_pid.get(pid)
        if sg is None:
            rows.append(feat)
            continue

        ta = sg['timestamp'].values

        # Define windows
        wa    = sg.loc[(ta >= ts - WINDOW_MS) & (ta <= ts),             SENSOR_COLS]
        wf    = sg.loc[(ta >= ts - WINDOW_MS) & (ta < ts - HALF_MS),    SENSOR_COLS]
        wl    = sg.loc[(ta >= ts - HALF_MS)   & (ta <= ts),             SENSOR_COLS]
        wt1   = sg.loc[(ta >= ts - WINDOW_MS) & (ta < ts - 2*THIRD_MS), SENSOR_COLS]
        wt3   = sg.loc[(ta >= ts - THIRD_MS)  & (ta <= ts),             SENSOR_COLS]
        wshrt = sg.loc[(ta >= ts - SHORT_MS)  & (ta <= ts),             SENSOR_COLS]
        # Lag window: 3–6 minutes BEFORE the current window  (NEW)
        wlag  = sg.loc[(ta >= ts - WINDOW_MS - LAG_MS) & (ta < ts - WINDOW_MS), SENSOR_COLS]

        feat['window_count'] = len(wa)

        # ── Standard per-sensor statistics ────────────────────────────────
        for c in SENSOR_COLS:
            v   = wa[c].dropna().values.astype(float)
            vf  = wf[c].dropna().values.astype(float)
            vl  = wl[c].dropna().values.astype(float)
            vt1 = wt1[c].dropna().values.astype(float)
            vt3 = wt3[c].dropna().values.astype(float)
            null_keys = ['mean','std','min','max','median','skew','kurt','range',
                         'q25','q75','iqr','delta','slope','t1_mean','t3_mean',
                         't3t1','p10','p90']
            if len(v) == 0:
                for s in null_keys:
                    feat[f'{c}_{s}'] = np.nan
                continue
            feat[f'{c}_mean']    = float(np.mean(v))
            feat[f'{c}_std']     = float(np.std(v))
            feat[f'{c}_min']     = float(np.min(v))
            feat[f'{c}_max']     = float(np.max(v))
            feat[f'{c}_median']  = float(np.median(v))
            feat[f'{c}_skew']    = float(spstats.skew(v))       if len(v) > 2 else 0.0
            feat[f'{c}_kurt']    = float(spstats.kurtosis(v))   if len(v) > 2 else 0.0
            feat[f'{c}_range']   = float(np.max(v) - np.min(v))
            feat[f'{c}_q25']     = float(np.percentile(v, 25))
            feat[f'{c}_q75']     = float(np.percentile(v, 75))
            feat[f'{c}_p10']     = float(np.percentile(v, 10))  # NEW
            feat[f'{c}_p90']     = float(np.percentile(v, 90))  # NEW
            feat[f'{c}_iqr']     = feat[f'{c}_q75'] - feat[f'{c}_q25']
            feat[f'{c}_delta']   = (float(np.mean(vl) - np.mean(vf))
                                    if len(vf) and len(vl) else 0.0)
            feat[f'{c}_slope']   = (float(np.polyfit(np.linspace(0, 1, len(v)), v, 1)[0])
                                    if len(v) > 2 else 0.0)
            feat[f'{c}_t1_mean'] = float(np.mean(vt1)) if len(vt1) else float(np.mean(v))
            feat[f'{c}_t3_mean'] = float(np.mean(vt3)) if len(vt3) else float(np.mean(v))
            feat[f'{c}_t3t1']    = feat[f'{c}_t3_mean'] - feat[f'{c}_t1_mean']

        # ── Accel magnitude ────────────────────────────────────────────────
        ax, ay, az = wa['accel_x'].values, wa['accel_y'].values, wa['accel_z'].values
        if len(ax):
            mag = np.sqrt(ax**2 + ay**2 + az**2)
            feat['accel_mag_mean'] = float(np.mean(mag))
            feat['accel_mag_std']  = float(np.std(mag))
            feat['accel_mag_max']  = float(np.max(mag))
        else:
            feat['accel_mag_mean'] = feat['accel_mag_std'] = feat['accel_mag_max'] = np.nan

        # ── HRV features ───────────────────────────────────────────────────
        feat.update(hrv_time_domain(wa['heart_rate']))
        feat.update(hrv_frequency_domain(wa['heart_rate']))

        # ── EDA peaks ──────────────────────────────────────────────────────
        feat.update(eda_peak_features(wa['eda']))

        # ── 1-min short window (HR + EDA) ──────────────────────────────────
        for c in ['heart_rate', 'eda']:
            vs = wshrt[c].dropna().values.astype(float)
            if len(vs) == 0:
                for s in ['short_mean', 'short_std', 'short_max', 'short_slope']:
                    feat[f'{c}_{s}'] = np.nan
            else:
                feat[f'{c}_short_mean']  = float(np.mean(vs))
                feat[f'{c}_short_std']   = float(np.std(vs))
                feat[f'{c}_short_max']   = float(np.max(vs))
                feat[f'{c}_short_slope'] = (float(np.polyfit(np.linspace(0, 1, len(vs)), vs, 1)[0])
                                            if len(vs) > 2 else 0.0)

        # ── Lag features — 3–6 min before current window (NEW) ────────────
        for c in ['heart_rate', 'eda', 'temperature']:
            vlag = wlag[c].dropna().values.astype(float)
            cur  = feat.get(f'{c}_mean', np.nan)
            if len(vlag) == 0:
                feat[f'{c}_lag_mean']  = np.nan
                feat[f'{c}_lag_std']   = np.nan
                feat[f'{c}_delta_lag'] = np.nan
            else:
                lag_mean = float(np.mean(vlag))
                feat[f'{c}_lag_mean']  = lag_mean
                feat[f'{c}_lag_std']   = float(np.std(vlag))
                feat[f'{c}_delta_lag'] = (cur - lag_mean) if np.isfinite(cur) else np.nan

        # ── Resting-baseline deviation ─────────────────────────────────────
        ref = refs.get(pid, {})
        if ref:
            hr_m   = feat.get('heart_rate_mean', np.nan)
            eda_m  = feat.get('eda_mean', np.nan)
            temp_m = feat.get('temperature_mean', np.nan)
            feat['hr_dev_rest']      = (hr_m   - ref['hr'])                    if np.isfinite(hr_m)   else np.nan
            feat['hr_dev_rest_std']  = (hr_m   - ref['hr'])   / ref['hr_std']  if np.isfinite(hr_m)   else np.nan
            feat['eda_dev_rest']     = (eda_m  - ref['eda'])                   if np.isfinite(eda_m)  else np.nan
            feat['eda_dev_rest_std'] = (eda_m  - ref['eda'])  / ref['eda_std'] if np.isfinite(eda_m)  else np.nan
            feat['temp_dev_rest']    = (temp_m - ref['temp'])                  if np.isfinite(temp_m) else np.nan
            hd = feat.get('hr_dev_rest_std', np.nan)
            ed = feat.get('eda_dev_rest_std', np.nan)
            feat['compound_stress'] = (hd + ed) if (np.isfinite(hd) and np.isfinite(ed)) else np.nan
        else:
            for k in ['hr_dev_rest', 'hr_dev_rest_std', 'eda_dev_rest',
                      'eda_dev_rest_std', 'temp_dev_rest', 'compound_stress']:
                feat[k] = np.nan

        # ── Cross-channel correlations ──────────────────────────────────────
        try:
            hr   = wa['heart_rate'].dropna().values.astype(float)
            eda  = wa['eda'].dropna().values.astype(float)
            temp = wa['temperature'].dropna().values.astype(float)
            n_min = min(len(hr), len(eda), len(temp))
            if n_min >= 30:
                h, e, t = hr[:n_min], eda[:n_min], temp[:n_min]
                feat['corr_hr_eda']   = float(np.corrcoef(h, e)[0, 1]) if h.std() > 1e-6 and e.std() > 1e-6 else 0.0
                feat['corr_hr_temp']  = float(np.corrcoef(h, t)[0, 1]) if h.std() > 1e-6 and t.std() > 1e-6 else 0.0
                feat['corr_eda_temp'] = float(np.corrcoef(e, t)[0, 1]) if e.std() > 1e-6 and t.std() > 1e-6 else 0.0
            else:
                feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan
        except Exception:
            feat['corr_hr_eda'] = feat['corr_hr_temp'] = feat['corr_eda_temp'] = np.nan

        # ── PID encoding ───────────────────────────────────────────────────
        feat['pid_enc'] = pid_enc_map.get(pid, -1)

        rows.append(feat)
        if n % 200 == 0:
            print(f'  {n}/{len(label_df_e)} done')

    return pd.DataFrame(rows).set_index('id')


train_pid_map = {p: i for i, p in enumerate(TRAIN_LABEL['pid'].unique())}

print('\nExtracting train features...')
train_features = extract_features(TRAIN_LABEL_E, TRAIN_DATA, train_pid_map, TRAIN_REFS)
print('Extracting test features...')
test_features  = extract_features(TEST_LABEL_E,  TEST_DATA,  train_pid_map, TEST_REFS)
print(f'\ntrain: {train_features.shape}  |  test: {test_features.shape}')

new_cols = [c for c in train_features.columns
            if any(s in c for s in ['lag_', 'delta_lag', 'session_', 'p10', 'p90'])]
print(f'New features added ({len(new_cols)}): {new_cols}')


Extracting train features...
  200/815 done
  400/815 done
  600/815 done
  800/815 done
Extracting test features...
  200/1028 done
  400/1028 done
  600/1028 done
  800/1028 done
  1000/1028 done

train: (815, 157)  |  test: (1028, 157)
New features added (25): ['session_elapsed_ms', 'session_position', 'session_label_idx', 'session_total_labels', 'accel_x_p10', 'accel_x_p90', 'accel_y_p10', 'accel_y_p90', 'accel_z_p10', 'accel_z_p90', 'eda_p10', 'eda_p90', 'heart_rate_p10', 'heart_rate_p90', 'temperature_p10', 'temperature_p90', 'heart_rate_lag_mean', 'heart_rate_lag_std', 'heart_rate_delta_lag', 'eda_lag_mean', 'eda_lag_std', 'eda_delta_lag', 'temperature_lag_mean', 'temperature_lag_std', 'temperature_delta_lag']


## 6. Prepare Feature Matrices

In [8]:
tli        = TRAIN_LABEL.set_index('id')
y          = tli.loc[train_features.index, 'stress'].astype(int)
pid_groups = tli.loc[train_features.index, 'pid']

common_cols    = [c for c in train_features.columns if c in test_features.columns]
train_features = train_features[common_cols]
test_features  = test_features[common_cols]

imputer    = SimpleImputer(strategy='median')
X_imp      = pd.DataFrame(imputer.fit_transform(train_features), columns=common_cols, index=train_features.index)
X_test_imp = pd.DataFrame(imputer.transform(test_features),      columns=common_cols, index=test_features.index)

counts        = Counter(y)
total         = len(y)
class_weights = {0: total / (3 * counts[0]),
                 1: min(total / (3 * counts[1]), 2.5),
                 2: total / (3 * counts[2])}
sample_weights = np.array([class_weights[int(yi)] for yi in y])
train_prior    = np.array([counts[i] / total for i in range(3)])

print(f'\nX_imp        : {X_imp.shape}')
print(f'Class weights: {dict((k, round(v, 3)) for k, v in class_weights.items())}')
print(f'Train prior  : {train_prior.round(3).tolist()}')


X_imp        : (815, 157)
Class weights: {0: 1.677, 1: 2.5, 2: 0.463}
Train prior  : [0.199, 0.081, 0.72]


## 7. Session Helper Functions

In [9]:
def make_session_groups(label_df, gap_ms=GAP_MS):
    labels = label_df.copy().reset_index(drop=True)
    labels['rowpos'] = np.arange(len(labels))
    out = []
    for pid, grp in labels.sort_values(['pid', 'timestamp']).groupby('pid', sort=False):
        ts   = grp['timestamp'].values.astype(float)
        sess = np.cumsum(np.r_[0, np.diff(ts) > gap_ms])
        for sid in np.unique(sess):
            out.append(grp['rowpos'].values[sess == sid])
    return out


def smooth_by_session(proba, sessions, strength=0.30):
    out = proba.copy()
    for idx in sessions:
        mean   = proba[idx].mean(axis=0, keepdims=True)
        out[idx] = (1 - strength) * proba[idx] + strength * mean
    return out


train_label_for_rows = TRAIN_LABEL.set_index('id').loc[X_imp.index].reset_index()
TRAIN_SESSIONS = make_session_groups(train_label_for_rows)
TEST_SESSIONS  = make_session_groups(TEST_LABEL)
print(f'Train sessions: {len(TRAIN_SESSIONS)} | Test sessions: {len(TEST_SESSIONS)}')

Train sessions: 67 | Test sessions: 106


## 8. LOPO — Honest CV + Calibration Data

Leave-One-Person-Out produces honest OOF probabilities used to optimize per-class alpha. Fixes the class-1 suppression bug in v22.

In [10]:
LGBM_PARAMS = dict(
    n_estimators=1000, learning_rate=0.02, num_leaves=127, max_depth=-1,
    min_child_samples=10, subsample=0.6, colsample_bytree=0.5,
    reg_alpha=0.3, reg_lambda=0.3,
    class_weight='balanced', objective='multiclass', num_class=3,
    n_jobs=-1, verbose=-1,
)

print('\n══ LOPO (honest CV + calibration data) ══')
lopo_oof_proba = np.zeros((len(X_imp), 3))

for tr_idx, val_idx in LeaveOneGroupOut().split(X_imp, y, pid_groups):
    pid_val = pid_groups.iloc[val_idx[0]]
    model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': RANDOM_SEED})
    model.fit(
        X_imp.iloc[tr_idx], y.iloc[tr_idx],
        sample_weight=sample_weights[tr_idx],
        eval_set=[(X_imp.iloc[val_idx], y.iloc[val_idx])],
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(-1)],
    )
    lopo_oof_proba[val_idx] = model.predict_proba(X_imp.iloc[val_idx])
    pred = np.argmax(lopo_oof_proba[val_idx], axis=1)
    ba   = balanced_accuracy_score(y.iloc[val_idx], pred)
    print(f'  PID {pid_val}: BA={ba:.4f}  dist={dict(Counter(pred))}')

lopo_ba_raw = balanced_accuracy_score(y, np.argmax(lopo_oof_proba, axis=1))
print(f'\nHonest LOPO BA (raw):   {lopo_ba_raw:.4f}')
print(f'LOPO raw class dist:    {dict(Counter(np.argmax(lopo_oof_proba, axis=1)))}')


══ LOPO (honest CV + calibration data) ══
  PID 43JW: BA=0.0000  dist={np.int64(1): 26, np.int64(0): 67}
  PID C8Q6: BA=0.5000  dist={np.int64(2): 151, np.int64(1): 1}
  PID DT5C: BA=0.3777  dist={np.int64(1): 62, np.int64(0): 28}
  PID F1ZM: BA=0.4739  dist={np.int64(0): 7, np.int64(2): 130}
  PID HDS9: BA=0.5043  dist={np.int64(0): 87, np.int64(1): 6, np.int64(2): 42}
  PID P4DZ: BA=0.3265  dist={np.int64(1): 143, np.int64(0): 1}
  PID TPQI: BA=0.2143  dist={np.int64(1): 23, np.int64(0): 41}

Honest LOPO BA (raw):   0.5546
LOPO raw class dist:    {np.int64(1): 261, np.int64(0): 231, np.int64(2): 323}


## 9. Per-Class Alpha Calibration *(KEY FIX)*

v22 uses `alpha=1.6` for ALL classes → class 1 collapses to ~10 predictions. We optimize `alpha_0`, `alpha_1`, `alpha_2` separately using LOPO OOF proba to maximize balanced accuracy.

In [11]:
def calibrate_proba(proba, alpha_vec, prior):
    """Per-class prior calibration: proba * prior^alpha_vec."""
    cal = proba * (prior ** np.array(alpha_vec))
    return cal / (cal.sum(axis=1, keepdims=True) + 1e-9)


def score_alpha(alpha_vec, proba, y_true, sessions, smooth=0.30):
    cal = calibrate_proba(proba, alpha_vec, train_prior)
    if smooth > 0:
        cal = smooth_by_session(cal, sessions, smooth)
    return balanced_accuracy_score(y_true, np.argmax(cal, axis=1))


print('\n══ Per-class alpha calibration via LOPO OOF ══')
y_np = y.values.astype(int)
best_alpha_vec = np.array([1.5, 1.5, 1.5])
best_cal_ba    = -np.inf

# Multiple starting points to escape local optima
# x0 chosen to span different class-1 alpha values (low alpha_1 = keep more class 1)
start_points = [
    [1.0, 0.3, 1.5],
    [1.2, 0.6, 1.8],
    [1.5, 0.8, 2.0],
    [1.6, 1.2, 2.0],
    [0.8, 0.2, 1.2],
    [1.4, 0.5, 1.6],
    [2.0, 1.0, 2.5],
    [1.0, 1.0, 1.0],   # uniform baseline
]

for x0 in start_points:
    res = minimize(
        lambda av: -score_alpha(av, lopo_oof_proba, y_np, TRAIN_SESSIONS, 0.30),
        x0, method='Nelder-Mead',
        options={'xatol': 0.003, 'fatol': 0.001, 'maxiter': 1000},
    )
    if -res.fun > best_cal_ba:
        best_cal_ba    = -res.fun
        best_alpha_vec = res.x

# Validate
lopo_ba_cal = score_alpha(best_alpha_vec, lopo_oof_proba, y_np, TRAIN_SESSIONS, 0.30)
cal_dist    = np.bincount(
    np.argmax(calibrate_proba(lopo_oof_proba, best_alpha_vec, train_prior), axis=1), minlength=3
)

print(f'Optimal alpha_vec:       {best_alpha_vec.round(3)}')
print(f'LOPO BA uncalibrated:    {lopo_ba_raw:.4f}')
print(f'LOPO BA + per-class cal: {lopo_ba_cal:.4f}')
print(f'LOPO calibrated dist:    {cal_dist.tolist()}  (target ~[{int(0.199*total)}, {int(0.081*total)}, {int(0.72*total)}])')


══ Per-class alpha calibration via LOPO OOF ══
Optimal alpha_vec:       [1.642 0.436 2.633]
LOPO BA uncalibrated:    0.5546
LOPO BA + per-class cal: 0.5333
LOPO calibrated dist:    [40, 336, 439]  (target ~[162, 66, 586])


## 10. Final 7-Seed LightGBM Ensemble

In [12]:
SEEDS    = [42, 7, 123, 17, 99, 256, 314]
N_SPLITS = 5

print('\n══ LightGBM: 7 seeds × 5 folds ══')
lgbm_test_proba = np.zeros((len(X_test_imp), 3))
lgbm_cv_scores  = []

for seed in SEEDS:
    skf        = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
    seed_proba = np.zeros((len(X_test_imp), 3))
    fold_scores = []
    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y), 1):
        m = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': seed})
        m.fit(
            X_imp.iloc[tr_idx], y.iloc[tr_idx],
            sample_weight=sample_weights[tr_idx],
            eval_set=[(X_imp.iloc[val_idx], y.iloc[val_idx])],
            callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(-1)],
        )
        fold_scores.append(balanced_accuracy_score(y.iloc[val_idx], m.predict(X_imp.iloc[val_idx])))
        seed_proba += m.predict_proba(X_test_imp)
    seed_proba /= N_SPLITS
    lgbm_test_proba += seed_proba
    lgbm_cv_scores.append(np.mean(fold_scores))
    print(f'  Seed {seed}: leaky CV={np.mean(fold_scores):.4f}')

lgbm_test_proba /= len(SEEDS)
print(f'\nLGBM mean leaky CV: {np.mean(lgbm_cv_scores):.4f}')


══ LightGBM: 7 seeds × 5 folds ══
  Seed 42: leaky CV=0.9185
  Seed 7: leaky CV=0.9101
  Seed 123: leaky CV=0.9009
  Seed 17: leaky CV=0.8806
  Seed 99: leaky CV=0.9030
  Seed 256: leaky CV=0.8835
  Seed 314: leaky CV=0.9070

LGBM mean leaky CV: 0.9005


## 11. Optional XGBoost Ensemble

Adds model diversity — different tree-building algorithm. Blended 60% LGBM / 40% XGB.

In [13]:
if HAS_XGB:
    print('\n══ XGBoost: 7 seeds × 5 folds ══')
    XGB_PARAMS = dict(
        n_estimators=600, learning_rate=0.02, max_depth=6,
        subsample=0.7, colsample_bytree=0.6,
        reg_alpha=0.3, reg_lambda=1.0,
        eval_metric='mlogloss',
        early_stopping_rounds=50,  # moved here from fit() for XGBoost >= 2.0
        n_jobs=-1, verbosity=0,
    )
    xgb_test_proba = np.zeros((len(X_test_imp), 3))
    xgb_cv_scores  = []

    for seed in SEEDS:
        skf        = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=seed)
        seed_proba = np.zeros((len(X_test_imp), 3))
        fold_scores = []
        for fold, (tr_idx, val_idx) in enumerate(skf.split(X_imp, y), 1):
            m = xgb.XGBClassifier(**{**XGB_PARAMS, 'random_state': seed})
            m.fit(
                X_imp.iloc[tr_idx], y.iloc[tr_idx],
                sample_weight=sample_weights[tr_idx],
                eval_set=[(X_imp.iloc[val_idx], y.iloc[val_idx])],
                verbose=False,
            )
            fold_scores.append(balanced_accuracy_score(y.iloc[val_idx], m.predict(X_imp.iloc[val_idx])))
            seed_proba += m.predict_proba(X_test_imp)
        seed_proba /= N_SPLITS
        xgb_test_proba += seed_proba
        xgb_cv_scores.append(np.mean(fold_scores))
        print(f'  Seed {seed}: leaky CV={np.mean(fold_scores):.4f}')

    xgb_test_proba /= len(SEEDS)
    print(f'\nXGB mean leaky CV: {np.mean(xgb_cv_scores):.4f}')
    raw_test_proba = 0.60 * lgbm_test_proba + 0.40 * xgb_test_proba
    print('Final proba: 60% LGBM + 40% XGB blend')
else:
    raw_test_proba = lgbm_test_proba
    print('\nUsing LGBM-only probabilities')

print(f'Raw test argmax dist: {dict(Counter(np.argmax(raw_test_proba, axis=1)))}')


══ XGBoost: 7 seeds × 5 folds ══
  Seed 42: leaky CV=0.8699
  Seed 7: leaky CV=0.8610
  Seed 123: leaky CV=0.8566
  Seed 17: leaky CV=0.8495
  Seed 99: leaky CV=0.8688
  Seed 256: leaky CV=0.8511
  Seed 314: leaky CV=0.8682

XGB mean leaky CV: 0.8607
Final proba: 60% LGBM + 40% XGB blend
Raw test argmax dist: {np.int64(2): 418, np.int64(1): 214, np.int64(0): 396}


## 12. Calibrate & Generate Submissions

In [14]:
def make_submission(proba, alpha_vec, smooth, sessions, prior, fname):
    cal   = calibrate_proba(proba, alpha_vec, prior)
    if smooth > 0:
        cal = smooth_by_session(cal, sessions, strength=smooth)
    preds  = np.argmax(cal, axis=1).astype(int)
    cnt    = np.bincount(preds, minlength=3)
    fracs  = cnt / len(preds)
    dev    = np.abs(fracs - prior).max()
    pd.DataFrame({'id': TEST_LABEL['id'].values, 'stress': preds}).to_csv(fname, index=False)
    return preds, cnt, fracs, dev


print('\n┌────────────────────────────────────────────────────────────────────┐')
print('│                       Submission variants                          │')
print('├──────────────────────────────┬──────────────────┬───────────┬──────┤')
print('│ File                         │ alpha_vec        │ dist      │ dev  │')
print('├──────────────────────────────┼──────────────────┼───────────┼──────┤')

# A — per-class optimized (PRIMARY)
p_opt, c_opt, f_opt, d_opt = make_submission(
    raw_test_proba, best_alpha_vec, 0.30, TEST_SESSIONS, train_prior, 'submission_opt.csv')
print(f'│ submission_opt.csv           │ {str(best_alpha_vec.round(2)):16s} │ {str(c_opt.tolist()):9s} │ {d_opt:.3f} │  ← RECOMMENDED')

# B — uniform alpha=1.4
p14, c14, f14, d14 = make_submission(
    raw_test_proba, [1.4, 1.4, 1.4], 0.30, TEST_SESSIONS, train_prior, 'submission_alpha_1.4.csv')
print(f'│ submission_alpha_1.4.csv     │ [1.4, 1.4, 1.4] │ {str(c14.tolist()):9s} │ {d14:.3f} │')

# C — uniform alpha=1.6 (v22 default)
p16, c16, f16, d16 = make_submission(
    raw_test_proba, [1.6, 1.6, 1.6], 0.30, TEST_SESSIONS, train_prior, 'submission_alpha_1.6.csv')
print(f'│ submission_alpha_1.6.csv     │ [1.6, 1.6, 1.6] │ {str(c16.tolist()):9s} │ {d16:.3f} │  (v22 default)')

print('└──────────────────────────────┴──────────────────┴───────────┴──────┘')

# Copy best as default
shutil.copy('submission_opt.csv', 'submission.csv')

print(f'\n>>> DEFAULT submission.csv = per-class optimized calibration')
print(f'    alpha_vec   : {best_alpha_vec.round(3)}')
print(f'    dist (0/1/2): {c_opt.tolist()}')
print(f'    fracs       : {f_opt.round(3).tolist()}')
print(f'    train prior : {train_prior.round(3).tolist()}')
print(f'    target      : ~[0.199, 0.081, 0.72]')


┌────────────────────────────────────────────────────────────────────┐
│                       Submission variants                          │
├──────────────────────────────┬──────────────────┬───────────┬──────┤
│ File                         │ alpha_vec        │ dist      │ dev  │
├──────────────────────────────┼──────────────────┼───────────┼──────┤
│ submission_opt.csv           │ [1.64 0.44 2.63] │ [26, 264, 738] │ 0.176 │  ← RECOMMENDED
│ submission_alpha_1.4.csv     │ [1.4, 1.4, 1.4] │ [57, 13, 958] │ 0.212 │
│ submission_alpha_1.6.csv     │ [1.6, 1.6, 1.6] │ [31, 3, 994] │ 0.247 │  (v22 default)
└──────────────────────────────┴──────────────────┴───────────┴──────┘

>>> DEFAULT submission.csv = per-class optimized calibration
    alpha_vec   : [1.642 0.436 2.633]
    dist (0/1/2): [26, 264, 738]
    fracs       : [0.025, 0.257, 0.718]
    train prior : [0.199, 0.081, 0.72]
    target      : ~[0.199, 0.081, 0.72]


## 13. Feature Importance

In [15]:
last_model = lgb.LGBMClassifier(**{**LGBM_PARAMS, 'random_state': 42})
last_model.fit(X_imp, y, sample_weight=sample_weights, callbacks=[lgb.log_evaluation(-1)])
imp = pd.Series(last_model.feature_importances_, index=X_imp.columns).sort_values(ascending=False)

print('\nTop 30 features by importance:')
NEW_TAGS = ['lag_', 'delta_lag', 'session_', 'p10', 'p90']
for name, val in imp.head(30).items():
    tag = '  ← NEW' if any(t in name for t in NEW_TAGS) else ''
    print(f'  {name:35s}: {int(val):5d}{tag}')


Top 30 features by importance:
  session_total_labels               :  1846  ← NEW
  temp_dev_rest                      :  1332
  pid_enc                            :   959
  eda_dev_rest_std                   :   479
  temperature_skew                   :   421
  session_elapsed_ms                 :   417  ← NEW
  heart_rate_min                     :   396
  temperature_max                    :   363
  eda_skew                           :   363
  temperature_lag_mean               :   325  ← NEW
  eda_t1_mean                        :   305
  eda_dev_rest                       :   303
  temperature_mean                   :   289
  compound_stress                    :   284
  accel_mag_std                      :   283
  eda_min                            :   275
  temperature_min                    :   271
  accel_z_mean                       :   268
  accel_z_t1_mean                    :   263
  accel_mag_mean                     :   257
  accel_z_t3_mean                    :   250
  

## 14. Summary

In [16]:
print('\n══════════════ v23 SUMMARY ══════════════')
print(f'Honest LOPO BA (no cal)      : {lopo_ba_raw:.4f}')
print(f'Honest LOPO BA (per-class α) : {lopo_ba_cal:.4f}   ← use this as real estimate')
print(f'Leaky SKF CV BA              : {np.mean(lgbm_cv_scores):.4f}   (inflated — ignore)')
print(f'Optimal alpha_vec            : {best_alpha_vec.round(3)}')
print()
print('Files saved:')
print('  submission.csv            ← DEFAULT (per-class optimized α)')
print('  submission_opt.csv        ← same as default')
print('  submission_alpha_1.4.csv  ← uniform α=1.4 fallback')
print('  submission_alpha_1.6.csv  ← v22-style fallback')
print()
print('SUBMIT: submission.csv (optimized) first.')
print('        If LB drops vs v22, try submission_alpha_1.4.csv.')
print('═════════════════════════════════════════')


══════════════ v23 SUMMARY ══════════════
Honest LOPO BA (no cal)      : 0.5546
Honest LOPO BA (per-class α) : 0.5333   ← use this as real estimate
Leaky SKF CV BA              : 0.9005   (inflated — ignore)
Optimal alpha_vec            : [1.642 0.436 2.633]

Files saved:
  submission.csv            ← DEFAULT (per-class optimized α)
  submission_opt.csv        ← same as default
  submission_alpha_1.4.csv  ← uniform α=1.4 fallback
  submission_alpha_1.6.csv  ← v22-style fallback

SUBMIT: submission.csv (optimized) first.
        If LB drops vs v22, try submission_alpha_1.4.csv.
═════════════════════════════════════════
